# Robosuite Scripted Data Collection + SmolVLA Training & Evaluation

End-to-end pipeline for 6 robosuite manipulation tasks:

| # | Environment | Description | Target Demos |
|---|---|---|---|
| 1 | **Lift** | Pick up a cube and lift it | 50 |
| 2 | **Stack** | Pick red cube, stack on green cube | 50 |
| 3 | **PickPlaceSingle** | Pick object, place in target bin | 50 |
| 4 | **NutAssemblySquare** | Place square nut on square peg | 75 |
| 5 | **NutAssemblyRound** | Place round nut on round peg | 75 |
| 6 | **NutAssembly** | Both nuts on their pegs | 75 |

## Pipeline
1. **Setup** — Install deps, configure EGL rendering
2. **Scripted Policies** — Proper hover -> descend -> grip -> lift -> move -> place sequences
3. **Trial Runs** — 2 episodes per scenario with video (verify before full run)
4. **Full Collection** — HDF5 demos (50 each, 75 for nut assembly)
5. **Convert to LeRobot** — HDF5 -> LeRobot v3.0 format for SmolVLA training
6. **Train SmolVLA** — Fine-tune on collected demos
7. **Evaluate VLA** — Test trained model on all 6 envs with video + success reporting

**Requirements:** GPU runtime (Runtime -> Change runtime type -> T4 or A100 GPU)

---
## 1. System Setup & Dependencies

In [ ]:
%%bash
# Install system dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# Create NVIDIA EGL ICD config (Colab is missing this by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
# Install Python packages
# 1) Simulation + data
!pip install -q robosuite imageio[ffmpeg] matplotlib h5py Pillow pandas

# 2) LeRobot with SmolVLA support (for training + evaluation)
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"

# 3) Pin numpy (mujoco needs >=2.0; numba needs <2.1)
!pip install -q numpy==2.0.2

print("\nPackages installed. Restarting runtime to fix numpy C bindings...")
print("After restart, SKIP this cell and continue from the next section.")

import os
os.kill(os.getpid(), 9)

### After runtime restart -- continue from here

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

import numpy as np
import robosuite as suite
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import base64
import h5py
import time
import json
import torch

print(f"robosuite {robosuite.__version__}")
print(f"numpy {np.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete.")

---
## 2. Scripted Policies

Each policy uses a **state-machine** approach with proper phases:
1. **Hover** -- move above the target object (open gripper)
2. **Descend** -- lower onto the object
3. **Grasp** -- close gripper and wait for it to firmly grip
4. **Lift** -- raise the object
5. **Move** -- transport to target location (for place/stack tasks)
6. **Place/Release** -- open gripper at target + retreat

Gripper convention: **+1 = close, -1 = open** (verified from robosuite Panda source).

In [ ]:
# ============================================================
# SCRIPTED POLICIES -- All 6 Scenarios
# ============================================================
#
# Gripper: +1 = close, -1 = open (robosuite PandaGripper)
#
# Action space (7D OSC_POSE for Panda):
#   [dx, dy, dz, dax, day, daz, gripper]
#   All values clipped to [-1, 1]
# ============================================================

TABLE_HEIGHT = 0.8
HOVER_DELTA_Z = 0.12     # hover this far above the object
GRASP_XY_THRESH = 0.01   # close enough in XY to descend
GRIP_WAIT_STEPS = 15     # steps to wait while gripper closes
GAIN = 8.0               # proportional gain for position control


class LiftPolicy:
    """Lift: hover above cube -> descend -> grip -> lift.
    Success: cube_z > table_height + 0.04
    Obs keys: cube_pos, robot0_eef_pos
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0

    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        cube = obs["cube_pos"]
        action = np.zeros(7)

        if self.phase == "hover":
            target = cube.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = cube.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = cube.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1  # close
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1  # keep closed

        return np.clip(action, -1, 1)


class StackPolicy:
    """Stack: pick cubeA (red) -> place on cubeB (green).
    Success: cubeA on cubeB, not grasped
    Obs keys: cubeA_pos, cubeB_pos, robot0_eef_pos
    """
    def __init__(self):
        self.phase = "hover_A"
        self.grip_counter = 0
        self.release_counter = 0

    def reset(self):
        self.phase = "hover_A"
        self.grip_counter = 0
        self.release_counter = 0

    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        cubeA = obs["cubeA_pos"]
        cubeB = obs["cubeB_pos"]
        action = np.zeros(7)

        if self.phase == "hover_A":
            target = cubeA.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend_A"

        elif self.phase == "descend_A":
            target = cubeA.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip_A"
                self.grip_counter = 0

        elif self.phase == "grip_A":
            target = cubeA.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift_A"

        elif self.phase == "lift_A":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > cubeB[2] + 0.15:
                self.phase = "hover_B"

        elif self.phase == "hover_B":
            target = cubeB.copy()
            target[2] += 0.10
            delta = target - ee
            action[:3] = delta * GAIN * 0.6
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.015 and abs(delta[2]) < 0.02:
                self.phase = "descend_B"

        elif self.phase == "descend_B":
            target = cubeB.copy()
            target[2] += 0.05
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta) < 0.015:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0

        return np.clip(action, -1, 1)


class PickPlaceSinglePolicy:
    """PickPlaceSingle: pick object -> place in bin2.
    Success: object in bin2 AND gripper backed away (r_reach < 0.6)
    Obs keys: Milk_pos / Bread_pos / Cereal_pos / Can_pos, robot0_eef_pos
    Bin2 at (0.1, 0.28, 0.8)
    """
    OBJ_NAMES = ["Milk", "Bread", "Cereal", "Can"]

    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.active_obj = None

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.active_obj = None

    def _find_object(self, obs):
        if self.active_obj is not None:
            return self.active_obj
        for name in self.OBJ_NAMES:
            if f"{name}_pos" in obs:
                self.active_obj = name
                return name
        return None

    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        obj_name = self._find_object(obs)
        if obj_name is None:
            return np.zeros(7)
        obj_pos = obs[f"{obj_name}_pos"]
        bin2 = np.array([0.1, 0.28, 0.8])
        action = np.zeros(7)

        if self.phase == "hover":
            target = obj_pos.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = obj_pos.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = obj_pos.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > bin2[2] + 0.20:
                self.phase = "move_to_bin"

        elif self.phase == "move_to_bin":
            target = bin2.copy()
            target[2] = ee[2]
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.03:
                self.phase = "lower_to_bin"

        elif self.phase == "lower_to_bin":
            target = bin2.copy()
            target[2] += 0.10
            delta = target - ee
            action[:3] = delta * GAIN * 0.4
            action[6] = 1
            if abs(delta[2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0  # retreat up so gripper is far from object

        return np.clip(action, -1, 1)


class NutAssemblySquarePolicy:
    """NutAssemblySquare: pick square nut -> place on square peg.
    Success: nut within 0.03m xy of peg AND z < table+0.05 AND gripper away
    Obs keys: SquareNut_pos, robot0_eef_pos. Peg pos from env.sim.
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def __call__(self, obs, peg_pos=None):
        ee = obs["robot0_eef_pos"]
        nut = obs["SquareNut_pos"]
        action = np.zeros(7)
        if peg_pos is None:
            peg_pos = np.array([0.12, 0.12, 0.8])

        if self.phase == "hover":
            target = nut.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            target = peg_pos.copy()
            target[2] = ee[2]
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "lower_to_peg"

        elif self.phase == "lower_to_peg":
            target = peg_pos.copy()
            target[2] += 0.04
            delta = target - ee
            action[:3] = delta * GAIN * 0.3
            action[6] = 1
            if abs(delta[2]) < 0.02 and np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0  # retreat

        return np.clip(action, -1, 1)


class NutAssemblyRoundPolicy:
    """NutAssemblyRound: pick round nut -> place on round peg.
    Obs keys: RoundNut_pos, robot0_eef_pos. Peg2 from env.sim.
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def __call__(self, obs, peg_pos=None):
        ee = obs["robot0_eef_pos"]
        nut = obs["RoundNut_pos"]
        action = np.zeros(7)
        if peg_pos is None:
            peg_pos = np.array([0.12, -0.12, 0.8])

        if self.phase == "hover":
            target = nut.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            target = peg_pos.copy()
            target[2] = ee[2]
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "lower_to_peg"

        elif self.phase == "lower_to_peg":
            target = peg_pos.copy()
            target[2] += 0.04
            delta = target - ee
            action[:3] = delta * GAIN * 0.3
            action[6] = 1
            if abs(delta[2]) < 0.02 and np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0

        return np.clip(action, -1, 1)


class NutAssemblyFullPolicy:
    """NutAssembly (both): square nut -> peg1, then round nut -> peg2.
    Success: BOTH nuts placed.
    Obs keys: SquareNut_pos, RoundNut_pos, robot0_eef_pos.
    """
    def __init__(self):
        self.current_nut = "square"
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.retreat_counter = 0

    def reset(self):
        self.current_nut = "square"
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.retreat_counter = 0

    def __call__(self, obs, peg1_pos=None, peg2_pos=None):
        ee = obs["robot0_eef_pos"]
        action = np.zeros(7)

        if self.current_nut == "square":
            nut = obs["SquareNut_pos"]
            peg = peg1_pos if peg1_pos is not None else np.array([0.12, 0.12, 0.8])
        else:
            nut = obs["RoundNut_pos"]
            peg = peg2_pos if peg2_pos is not None else np.array([0.12, -0.12, 0.8])

        if self.phase == "hover":
            target = nut.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            target = peg.copy()
            target[2] = ee[2]
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "lower_to_peg"

        elif self.phase == "lower_to_peg":
            target = peg.copy()
            target[2] += 0.04
            delta = target - ee
            action[:3] = delta * GAIN * 0.3
            action[6] = 1
            if abs(delta[2]) < 0.02 and np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 15:
                self.phase = "retreat"
                self.retreat_counter = 0

        elif self.phase == "retreat":
            action[2] = 1.0
            action[6] = -1
            self.retreat_counter += 1
            if self.retreat_counter > 20:
                if self.current_nut == "square":
                    self.current_nut = "round"
                    self.phase = "hover"
                    self.grip_counter = 0
                    self.release_counter = 0
                    self.retreat_counter = 0
                else:
                    self.phase = "done"

        elif self.phase == "done":
            pass

        return np.clip(action, -1, 1)


print("All 6 scripted policies defined.")

---
## 3. Environment Config & Episode Runner

In [ ]:
# ============================================================
# Task descriptions for each scenario (used by SmolVLA)
# ============================================================
TASK_DESCRIPTIONS = {
    "Lift": "Pick up the cube and lift it off the table",
    "Stack": "Pick up the red cube and stack it on top of the green cube",
    "PickPlaceSingle": "Pick up the object and place it in the bin",
    "NutAssemblySquare": "Pick up the square nut and place it on the square peg",
    "NutAssemblyRound": "Pick up the round nut and place it on the round peg",
    "NutAssembly": "Assemble both the square nut and round nut onto their respective pegs",
}

SCENARIOS = {
    "Lift": {
        "env_kwargs": dict(
            env_name="Lift", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=400,
        ),
        "policy_class": LiftPolicy, "max_steps": 400, "target_demos": 50,
    },
    "Stack": {
        "env_kwargs": dict(
            env_name="Stack", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=500,
        ),
        "policy_class": StackPolicy, "max_steps": 500, "target_demos": 50,
    },
    "PickPlaceSingle": {
        "env_kwargs": dict(
            env_name="PickPlaceSingle", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=500,
        ),
        "policy_class": PickPlaceSinglePolicy, "max_steps": 500, "target_demos": 50,
    },
    "NutAssemblySquare": {
        "env_kwargs": dict(
            env_name="NutAssemblySquare", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=600,
        ),
        "policy_class": NutAssemblySquarePolicy, "max_steps": 600, "target_demos": 75,
    },
    "NutAssemblyRound": {
        "env_kwargs": dict(
            env_name="NutAssemblyRound", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=600,
        ),
        "policy_class": NutAssemblyRoundPolicy, "max_steps": 600, "target_demos": 75,
    },
    "NutAssembly": {
        "env_kwargs": dict(
            env_name="NutAssembly", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, single_object_mode=0, horizon=800,
        ),
        "policy_class": NutAssemblyFullPolicy, "max_steps": 800, "target_demos": 75,
    },
}


def _get_peg_positions(env, scenario_name):
    """Read peg positions from env sim data for nut assembly tasks."""
    if scenario_name == "NutAssemblySquare":
        return {"peg_pos": env.sim.data.body_xpos[env.peg1_body_id].copy()}
    elif scenario_name == "NutAssemblyRound":
        return {"peg_pos": env.sim.data.body_xpos[env.peg2_body_id].copy()}
    elif scenario_name == "NutAssembly":
        return {
            "peg1_pos": env.sim.data.body_xpos[env.peg1_body_id].copy(),
            "peg2_pos": env.sim.data.body_xpos[env.peg2_body_id].copy(),
        }
    return {}


def run_episode(env, policy, scenario_name, max_steps, record_video=False):
    """Run one episode with scripted policy.
    Returns dict: success, total_reward, steps, frames, actions, observations.
    Success is checked via env._check_success() (ground truth).
    """
    obs = env.reset()
    policy.reset()

    frames, actions_buf, obs_buf = [], [], []
    total_reward = 0.0
    success = False

    for step in range(max_steps):
        # Call policy with peg positions for nut assembly
        peg_kwargs = _get_peg_positions(env, scenario_name)
        action = policy(obs, **peg_kwargs)

        actions_buf.append(action.copy())

        # Store observation data for HDF5
        obs_entry = {}
        for key in ["agentview_image", "robot0_eye_in_hand_image",
                    "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]:
            if key in obs:
                obs_entry[key] = obs[key].copy()
        obs_buf.append(obs_entry)

        if record_video and "agentview_image" in obs:
            frames.append(np.flip(obs["agentview_image"], axis=0).copy())

        obs, reward, done, info = env.step(action)
        total_reward += reward

        if env._check_success():
            success = True

        if done:
            break

    if record_video and "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0).copy())

    return {
        "success": success,
        "total_reward": total_reward,
        "steps": len(actions_buf),
        "frames": frames,
        "actions": actions_buf,
        "observations": obs_buf,
    }


def save_video_file(frames, path, fps=20):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        writer.append_data(frame)
    writer.close()


def show_video_inline(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="512">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
        f'</video>'
    ))


print(f"Configured {len(SCENARIOS)} scenarios:")
for name, cfg in SCENARIOS.items():
    print(f"  {name}: max_steps={cfg['max_steps']}, target={cfg['target_demos']}")

---
## 4. TRIAL RUNS -- 2 Episodes Per Scenario (with Video)

Run 2 episodes per scenario with video. **Check each video before full collection.**
Each episode prints SUCCESS or FAIL with color-coded headers.

In [ ]:
TRIAL_EPISODES = 2
trial_results = {}

for scenario_name, cfg in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"  TRIAL: {scenario_name} ({TRIAL_EPISODES} episodes)")
    print(f"{'='*60}")

    env = suite.make(**cfg["env_kwargs"])
    policy = cfg["policy_class"]()
    results = []

    for ep in range(TRIAL_EPISODES):
        np.random.seed(42 + ep)
        result = run_episode(
            env, policy, scenario_name,
            max_steps=cfg["max_steps"],
            record_video=True,
        )
        results.append(result)

        status = "SUCCESS" if result["success"] else "FAIL"
        color = "green" if result["success"] else "red"
        print(f"  Episode {ep+1}: {status}  "
              f"(reward={result['total_reward']:.2f}, steps={result['steps']})")

        video_path = f"trial_videos/{scenario_name}_ep{ep+1}.mp4"
        if result["frames"]:
            save_video_file(result["frames"], video_path)
            display(HTML(
                f'<h4 style="color: {color}; border: 2px solid {color}; '
                f'padding: 4px 8px; display: inline-block;">'
                f'{scenario_name} | Episode {ep+1} | {status}</h4>'
            ))
            show_video_inline(video_path)

    env.close()

    n_success = sum(1 for r in results if r["success"])
    trial_results[scenario_name] = {
        "success": n_success,
        "total": TRIAL_EPISODES,
        "rate": n_success / TRIAL_EPISODES,
    }

# ============================================================
# TRIAL SUMMARY TABLE
# ============================================================
print(f"\n\n{'='*60}")
print(f"  TRIAL SUMMARY")
print(f"{'='*60}")
print(f"  {'Scenario':<25s} {'Result':>10s} {'Rate':>8s} {'Status':>8s}")
print(f"  {'-'*55}")
for name, r in trial_results.items():
    status = "PASS" if r["success"] > 0 else "FAIL"
    print(f"  {name:<25s} {r['success']}/{r['total']:>5d} {r['rate']:>7.0%} {status:>8s}")

all_pass = all(r["success"] > 0 for r in trial_results.values())
print(f"\n  {'ALL PASSED -- safe to proceed' if all_pass else 'SOME FAILED -- review videos above'}")

---
## 5. FULL DATA COLLECTION

Collect successful demos to HDF5. Only saves episodes where `env._check_success()` is True.
- Lift, Stack, PickPlaceSingle: **50** successful demos each
- NutAssemblySquare, NutAssemblyRound, NutAssembly: **75** each

In [ ]:
OUTPUT_DIR = "collected_demos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED_START = 100
MAX_ATTEMPT_MULTIPLIER = 3

collection_summary = {}

for scenario_name, cfg in SCENARIOS.items():
    target = cfg["target_demos"]
    max_attempts = target * MAX_ATTEMPT_MULTIPLIER

    print(f"\n{'='*70}")
    print(f"  COLLECTING: {scenario_name}")
    print(f"  Target: {target} successful demos | Max attempts: {max_attempts}")
    print(f"{'='*70}")

    env = suite.make(**cfg["env_kwargs"])
    policy = cfg["policy_class"]()

    hdf5_path = os.path.join(OUTPUT_DIR, f"{scenario_name.lower()}_demos.hdf5")
    h5_file = h5py.File(hdf5_path, "w")
    data_grp = h5_file.create_group("data")
    data_grp.attrs["env"] = scenario_name
    data_grp.attrs["policy"] = "scripted"
    data_grp.attrs["task_description"] = TASK_DESCRIPTIONS[scenario_name]

    successes = 0
    attempts = 0
    start_time = time.time()

    for ep in range(max_attempts):
        if successes >= target:
            break

        np.random.seed(SEED_START + ep)
        result = run_episode(
            env, policy, scenario_name,
            max_steps=cfg["max_steps"],
            record_video=False,
        )
        attempts += 1

        if result["success"]:
            demo_grp = data_grp.create_group(f"demo_{successes}")
            demo_grp.attrs["seed"] = SEED_START + ep
            demo_grp.attrs["num_samples"] = len(result["actions"])
            demo_grp.attrs["total_reward"] = result["total_reward"]

            demo_grp.create_dataset("actions", data=np.array(result["actions"]))

            obs_grp = demo_grp.create_group("obs")
            if result["observations"]:
                for key in result["observations"][0].keys():
                    obs_data = np.array([o[key] for o in result["observations"]])
                    obs_grp.create_dataset(key, data=obs_data)

            h5_file.flush()
            successes += 1

        if attempts % 10 == 0 or successes >= target:
            elapsed = time.time() - start_time
            rate = successes / max(attempts, 1)
            print(f"  [{attempts:4d} attempts]  {successes}/{target} demos  "
                  f"(rate: {rate:.1%})  {elapsed:.0f}s")

    env.close()

    data_grp.attrs["total_demos"] = successes
    data_grp.attrs["total_attempts"] = attempts
    data_grp.attrs["success_rate"] = successes / max(attempts, 1)
    h5_file.close()

    elapsed = time.time() - start_time
    collection_summary[scenario_name] = {
        "successes": successes, "target": target,
        "attempts": attempts, "rate": successes / max(attempts, 1),
        "hdf5_path": hdf5_path, "elapsed_s": elapsed,
    }

    status = "DONE" if successes >= target else "INCOMPLETE"
    print(f"  {status}: {successes}/{target} in {attempts} attempts ({elapsed:.0f}s)")

# Final summary
print(f"\n\n{'='*70}")
print(f"  DATA COLLECTION SUMMARY")
print(f"{'='*70}")
print(f"  {'Scenario':<25s} {'Collected':>10s} {'Attempts':>10s} {'Rate':>8s} {'Time':>8s}")
print(f"  {'-'*65}")
for name, r in collection_summary.items():
    print(f"  {name:<25s} {r['successes']:>5d}/{r['target']:<4d}  "
          f"{r['attempts']:>9d}  {r['rate']:>7.1%}  {r['elapsed_s']:>6.0f}s")

print(f"\n  HDF5 files in: {OUTPUT_DIR}/")
for name, r in collection_summary.items():
    print(f"    {r['hdf5_path']}  ({r['successes']} demos)")

---
## 6. Convert HDF5 to LeRobot v3.0 Format

Convert collected HDF5 demos into the format SmolVLA expects:
- **Parquet** files with per-frame rows (actions, states, episode_index, timestamps)
- **MP4 videos** for each episode (agentview + wrist camera)
- **Metadata** JSON

SmolVLA training expects these exact keys:
- `observation.images.image` -- agentview camera (256x256)
- `observation.images.image2` -- wrist camera (256x256)
- `observation.state` -- [eef_pos(3), eef_quat(4)] = 7D
- `action` -- [dx, dy, dz, dax, day, daz, gripper] = 7D

In [ ]:
import pandas as pd
from PIL import Image as PILImage

LEROBOT_DIR = "lerobot_dataset"

def convert_hdf5_to_lerobot(hdf5_path, scenario_name, output_base_dir):
    """Convert one HDF5 file to LeRobot v3.0 format."""
    output_dir = os.path.join(output_base_dir, scenario_name.lower())
    os.makedirs(os.path.join(output_dir, "data"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "videos", "agentview"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "videos", "wrist"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "meta"), exist_ok=True)

    task_desc = TASK_DESCRIPTIONS[scenario_name]
    action_names = ["dx", "dy", "dz", "dax", "day", "daz", "gripper"]

    all_rows = []
    episode_lengths = []

    with h5py.File(hdf5_path, "r") as f:
        demo_keys = sorted([k for k in f["data"].keys() if k.startswith("demo_")])
        print(f"  {scenario_name}: {len(demo_keys)} demos")

        for ep_idx, demo_key in enumerate(demo_keys):
            demo = f["data"][demo_key]
            actions = demo["actions"][:]
            n_steps = len(actions)
            episode_lengths.append(n_steps)

            # Extract obs
            has_agentview = "agentview_image" in demo["obs"]
            has_wrist = "robot0_eye_in_hand_image" in demo["obs"]
            eef_pos = demo["obs"]["robot0_eef_pos"][:] if "robot0_eef_pos" in demo["obs"] else np.zeros((n_steps, 3))
            eef_quat = demo["obs"]["robot0_eef_quat"][:] if "robot0_eef_quat" in demo["obs"] else np.zeros((n_steps, 4))

            # Save videos (flip + resize)
            if has_agentview:
                imgs = demo["obs"]["agentview_image"][:]
                vid_path = os.path.join(output_dir, "videos", "agentview", f"episode_{ep_idx:04d}.mp4")
                writer = imageio.get_writer(vid_path, fps=20)
                for frame in imgs:
                    frame = np.flip(frame, axis=0).copy()
                    if frame.shape[0] != 256:
                        frame = np.array(PILImage.fromarray(frame).resize((256, 256)))
                    writer.append_data(frame)
                writer.close()

            if has_wrist:
                imgs = demo["obs"]["robot0_eye_in_hand_image"][:]
                vid_path = os.path.join(output_dir, "videos", "wrist", f"episode_{ep_idx:04d}.mp4")
                writer = imageio.get_writer(vid_path, fps=20)
                for frame in imgs:
                    frame = np.flip(frame, axis=0).copy()
                    if frame.shape[0] != 256:
                        frame = np.array(PILImage.fromarray(frame).resize((256, 256)))
                    writer.append_data(frame)
                writer.close()

            # Build rows
            for t in range(n_steps):
                row = {
                    "episode_index": ep_idx,
                    "frame_index": t,
                    "timestamp": t / 20.0,
                    "task": task_desc,
                }
                # Actions
                for i, aname in enumerate(action_names):
                    row[f"action_{aname}"] = float(actions[t, i])
                # State: eef_pos + eef_quat
                for i in range(3):
                    row[f"robot0_eef_pos_{i}"] = float(eef_pos[t, i])
                for i in range(4):
                    row[f"robot0_eef_quat_{i}"] = float(eef_quat[t, i])
                all_rows.append(row)

    # Save parquet
    df = pd.DataFrame(all_rows)
    parquet_path = os.path.join(output_dir, "data", f"{scenario_name.lower()}.parquet")
    df.to_parquet(parquet_path, index=False)

    # Save metadata
    meta = {
        "task_name": scenario_name,
        "task_description": task_desc,
        "n_episodes": len(demo_keys),
        "episode_lengths": episode_lengths,
        "total_frames": sum(episode_lengths),
        "fps": 20,
        "action_dim": 7,
        "action_names": action_names,
        "state_dim": 7,
        "state_names": ["eef_pos_x", "eef_pos_y", "eef_pos_z",
                        "eef_quat_x", "eef_quat_y", "eef_quat_z", "eef_quat_w"],
        "image_size": 256,
        "cameras": ["agentview", "robot0_eye_in_hand"],
    }
    with open(os.path.join(output_dir, "meta", "info.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"    Saved: {len(df)} rows, {len(demo_keys)} episodes")
    print(f"    Parquet: {parquet_path}")
    print(f"    Videos: {output_dir}/videos/")
    return output_dir


# Convert all scenarios
print(f"\n{'='*60}")
print(f"  Converting HDF5 -> LeRobot v3.0 Format")
print(f"{'='*60}")

lerobot_paths = {}
for scenario_name, r in collection_summary.items():
    if r["successes"] > 0:
        lerobot_paths[scenario_name] = convert_hdf5_to_lerobot(
            r["hdf5_path"], scenario_name, LEROBOT_DIR
        )
    else:
        print(f"  {scenario_name}: SKIPPED (0 demos)")

print(f"\nConversion complete. LeRobot datasets in: {LEROBOT_DIR}/")

---
## 7. Train SmolVLA

Fine-tune SmolVLA on the collected demonstrations using `lerobot-train`.

**Estimated time:** ~3-4h on A100, ~6-8h on T4

You can train on individual scenarios or combine all data.

In [ ]:
# Auto-detect GPU and set batch size
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 4 if gpu_mem_gb > 30 else 2
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.0f}GB) -> batch_size={BATCH_SIZE}")
else:
    BATCH_SIZE = 1
    print("WARNING: No GPU detected")

# Pick which scenario to train on (change as needed)
TRAIN_SCENARIO = "Lift"  # Start with simplest
TRAIN_STEPS = 20000
CHECKPOINT_DIR = f"outputs/checkpoints/smolvla_{TRAIN_SCENARIO.lower()}"

train_dataset = lerobot_paths.get(TRAIN_SCENARIO)
if train_dataset:
    print(f"\nTraining SmolVLA on: {TRAIN_SCENARIO}")
    print(f"Dataset: {train_dataset}")
    print(f"Steps: {TRAIN_STEPS}, Batch: {BATCH_SIZE}")
    print(f"Output: {CHECKPOINT_DIR}")
else:
    print(f"No dataset for {TRAIN_SCENARIO}. Run collection first.")

In [ ]:
# Launch training
# NOTE: This uses the LeRobot v0.4+ CLI. Adjust repo_id to match your dataset.
if train_dataset:
    !lerobot-train \
        --policy.type=smolvla \
        --policy.load_vlm_weights=true \
        --dataset.repo_id={TRAIN_SCENARIO.lower()} \
        --dataset.root={train_dataset} \
        --batch_size={BATCH_SIZE} \
        --steps={TRAIN_STEPS} \
        --output_dir={CHECKPOINT_DIR} \
        --save_freq=2000 \
        --eval_freq=2000 \
        --log_freq=100 \
        --seed=42 \
        --policy.device=cuda

In [ ]:
# Post-training: fix n_action_steps for fast inference
import glob

config_files = glob.glob(os.path.join(CHECKPOINT_DIR, "**/config.json"), recursive=True)
for cfg_path in config_files:
    with open(cfg_path) as f:
        cfg = json.load(f)
    if cfg.get("n_action_steps") != 50:
        cfg["n_action_steps"] = 50
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Fixed n_action_steps=50 in {cfg_path}")
    else:
        print(f"Already correct: {cfg_path}")

---
## 8. Evaluate Trained VLA on All 6 Environments

Test the trained SmolVLA model on all 6 scenarios.
Every episode produces:
- **Video** (MP4) showing the robot's behavior
- **SUCCESS / FAIL** label (via `env._check_success()`)
- Per-scenario success rate summary table

In [ ]:
# ============================================================
# VLA EVALUATION HELPERS
# ============================================================

def load_vla_policy(checkpoint_path, device="cuda"):
    """Load trained SmolVLA policy."""
    try:
        from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
        policy = SmolVLAPolicy.from_pretrained(checkpoint_path)
        policy.to(device)
        policy.eval()
        print(f"Loaded VLA from {checkpoint_path}")
        return policy
    except Exception as e:
        print(f"Failed to load: {e}")
        print("Using random policy for testing.")
        return None


def get_vla_action(policy, obs, task_language, device="cuda"):
    """Get action from SmolVLA given robosuite observation."""
    if policy is None:
        return np.random.uniform(-0.3, 0.3, size=7)

    # Camera 1: agentview (flip + resize to 256x256)
    agentview = np.flip(
        obs.get("agentview_image", np.zeros((256, 256, 3), dtype=np.uint8)), axis=0
    ).copy()
    if agentview.shape[0] != 256:
        agentview = np.array(PILImage.fromarray(agentview).resize((256, 256)))

    # Camera 2: wrist camera
    wrist = obs.get("robot0_eye_in_hand_image", None)
    if wrist is not None:
        wrist = np.flip(wrist, axis=0).copy()
        if wrist.shape[0] != 256:
            wrist = np.array(PILImage.fromarray(wrist).resize((256, 256)))
    else:
        wrist = np.zeros((256, 256, 3), dtype=np.uint8)

    # State: eef_pos + eef_quat
    eef_pos = obs.get("robot0_eef_pos", np.zeros(3))
    eef_quat = obs.get("robot0_eef_quat", np.zeros(4))
    state = np.concatenate([eef_pos, eef_quat]).astype(np.float32)

    # SmolVLA input format
    obs_dict = {
        "observation.images.image": (
            torch.from_numpy(agentview).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
        ),
        "observation.images.image2": (
            torch.from_numpy(wrist).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
        ),
        "observation.state": torch.from_numpy(state).unsqueeze(0).to(device),
        "task": task_language,
    }

    with torch.no_grad():
        action = policy.select_action(obs_dict)

    if isinstance(action, torch.Tensor):
        action = action.cpu().numpy().flatten()

    return np.clip(action[:7], -1, 1)


def run_vla_episode(env, vla_policy, scenario_name, task_language, max_steps,
                    device="cuda", record_video=True):
    """Run one evaluation episode with VLA policy.
    Returns: success (bool), total_reward, steps, frames.
    """
    obs = env.reset()
    if vla_policy is not None:
        vla_policy.reset()  # clear action queue

    frames = []
    total_reward = 0.0
    success = False

    for step in range(max_steps):
        action = get_vla_action(vla_policy, obs, task_language, device=device)

        if record_video and "agentview_image" in obs:
            frames.append(np.flip(obs["agentview_image"], axis=0).copy())

        obs, reward, done, info = env.step(action)
        total_reward += reward

        if env._check_success():
            success = True

        if done:
            break

    if record_video and "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0).copy())

    return success, total_reward, step + 1, frames


print("VLA evaluation helpers loaded.")

In [ ]:
# ============================================================
# RUN VLA EVALUATION ON ALL 6 SCENARIOS
# ============================================================

EVAL_EPISODES = 10   # episodes per scenario for evaluation
EVAL_VIDEO_DIR = "eval_videos"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load trained policy
vla_policy = load_vla_policy(CHECKPOINT_DIR, device=DEVICE)

eval_results = {}

for scenario_name, cfg in SCENARIOS.items():
    task_lang = TASK_DESCRIPTIONS[scenario_name]
    max_steps = cfg["max_steps"]

    print(f"\n{'='*70}")
    print(f"  EVALUATING VLA: {scenario_name}")
    print(f"  Task: {task_lang}")
    print(f"  Episodes: {EVAL_EPISODES}, Max steps: {max_steps}")
    print(f"{'='*70}")

    env = suite.make(**cfg["env_kwargs"])
    ep_results = []

    for ep in range(EVAL_EPISODES):
        np.random.seed(1000 + ep)  # different seeds from collection

        success, reward, steps, frames = run_vla_episode(
            env, vla_policy, scenario_name, task_lang,
            max_steps=max_steps, device=DEVICE, record_video=True,
        )

        status = "SUCCESS" if success else "FAIL"
        color = "green" if success else "red"

        ep_results.append({
            "episode": ep, "success": success,
            "reward": reward, "steps": steps,
        })

        print(f"  Episode {ep+1:2d}: {status:7s}  "
              f"reward={reward:.2f}  steps={steps}")

        # Save video with success/fail in filename
        tag = "success" if success else "fail"
        video_path = os.path.join(
            EVAL_VIDEO_DIR, scenario_name,
            f"ep{ep+1:02d}_{tag}.mp4"
        )
        if frames:
            save_video_file(frames, video_path)

    env.close()

    n_success = sum(1 for r in ep_results if r["success"])
    rate = n_success / EVAL_EPISODES

    eval_results[scenario_name] = {
        "n_success": n_success,
        "n_episodes": EVAL_EPISODES,
        "success_rate": rate,
        "episodes": ep_results,
    }

    print(f"\n  {scenario_name} RESULT: {n_success}/{EVAL_EPISODES} ({rate:.0%})")

# ============================================================
# EVALUATION SUMMARY TABLE
# ============================================================
print(f"\n\n{'='*70}")
print(f"  VLA EVALUATION SUMMARY")
print(f"{'='*70}")
print(f"  {'Scenario':<25s} {'Success':>10s} {'Rate':>8s} {'Status':>10s}")
print(f"  {'-'*55}")

total_s, total_e = 0, 0
for name, r in eval_results.items():
    status = "PASS" if r["n_success"] > 0 else "FAIL"
    print(f"  {name:<25s} {r['n_success']:>3d}/{r['n_episodes']:<3d}  "
          f"{r['success_rate']:>7.0%} {status:>10s}")
    total_s += r["n_success"]
    total_e += r["n_episodes"]

print(f"  {'-'*55}")
print(f"  {'OVERALL':<25s} {total_s:>3d}/{total_e:<3d}  {total_s/total_e:>7.0%}")

# Save results JSON
os.makedirs("eval_results", exist_ok=True)
with open("eval_results/vla_eval.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print(f"\n  Results saved to eval_results/vla_eval.json")

In [ ]:
# ============================================================
# DISPLAY EVALUATION VIDEOS INLINE
# ============================================================
from pathlib import Path

for scenario_name in SCENARIOS.keys():
    video_dir = Path(EVAL_VIDEO_DIR) / scenario_name
    if not video_dir.exists():
        continue

    videos = sorted(video_dir.glob("*.mp4"))
    if not videos:
        continue

    r = eval_results[scenario_name]
    display(HTML(
        f'<h3>{scenario_name} -- '
        f'{r["n_success"]}/{r["n_episodes"]} ({r["success_rate"]:.0%})</h3>'
    ))

    for video_path in videos:
        name = video_path.stem
        is_success = "success" in name
        color = "green" if is_success else "red"
        status = "SUCCESS" if is_success else "FAIL"

        display(HTML(
            f'<span style="color: {color}; font-weight: bold;">'
            f'{name} -- {status}</span>'
        ))
        show_video_inline(str(video_path))

In [ ]:
# ============================================================
# SUCCESS/FAIL GRID VISUALIZATION
# ============================================================
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(eval_results), figsize=(3 * len(eval_results), 4))
if len(eval_results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, eval_results.items()):
    episodes = r["episodes"]
    cols = min(5, len(episodes))
    rows = (len(episodes) + cols - 1) // cols

    for i, ep in enumerate(episodes):
        row = i // cols
        col = i % cols
        color = '#4CAF50' if ep['success'] else '#f44336'
        rect = mpatches.FancyBboxPatch(
            (col, rows - 1 - row), 0.9, 0.9,
            boxstyle="round,pad=0.05", facecolor=color,
            edgecolor='white', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(col + 0.45, rows - 1 - row + 0.45, str(i + 1),
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    ax.set_xlim(-0.1, cols + 0.1)
    ax.set_ylim(-0.1, rows + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f"{name}\n{r['success_rate']:.0%}", fontsize=10, fontweight='bold')

success_patch = mpatches.Patch(color='#4CAF50', label='Success')
fail_patch = mpatches.Patch(color='#f44336', label='Fail')
fig.legend(handles=[success_patch, fail_patch], loc='lower center', ncol=2, fontsize=11)

plt.suptitle(f"VLA Evaluation: {total_s}/{total_e} ({total_s/total_e:.0%}) Overall",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("eval_results/eval_grid.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 9. (Optional) Save to Google Drive

In [ ]:
# Uncomment to save everything to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# drive_dir = "/content/drive/MyDrive/robosuite_vla"
# os.makedirs(drive_dir, exist_ok=True)
#
# # Copy demos, dataset, results, videos
# for src_dir in ["collected_demos", "lerobot_dataset", "eval_results",
#                 "eval_videos", "trial_videos"]:
#     if os.path.exists(src_dir):
#         shutil.copytree(src_dir, os.path.join(drive_dir, src_dir), dirs_exist_ok=True)
#
# # Copy checkpoint
# if os.path.exists(CHECKPOINT_DIR):
#     shutil.copytree(CHECKPOINT_DIR, os.path.join(drive_dir, "checkpoint"), dirs_exist_ok=True)
#
# print(f"All saved to {drive_dir}")